# 1. Dataset Verification and Feasibility Analysis
We start by inspecting our primary candidate dataset from Mendeley Data: *Predictive Structural Risk Evaluation Through Crack Imaging and Environmental Condition Metrics*.
Let's write a script to fetch the metadata or download URL to see what files are contained and if they have usable structured risk labels.

In [1]:
import urllib.request
import json
import os

# Mendeley Data API endpoint for dataset details
dataset_id = "j8z5cfcn33"
version = "1"
api_url = f"https://api.mendeley.com/datasets/{dataset_id}" # Note: Public API endpoint or direct download

print("Verifying dataset source...")
# Let's write a robust script to download and check the files directly from Mendeley's public download links.
direct_download_url = "https://data.mendeley.com/public-files/datasets/j8z5cfcn33/files/94f796a5-51db-40a2-a841-fcf67bf755b4/file_downloaded" # generic structure helper
print("We will download and inspect the contents programmatically.")

Verifying dataset source...
We will download and inspect the contents programmatically.


```markdown
## How to Run This on a Mobile Phone (Google Colab)
1. Tap the **Play** (▶️) button on the left of each code cell sequentially from top to bottom.
2. Wait for each cell to finish executing (indicated by a green checkmark) before moving to the next cell.
3. Keep this browser tab active to prevent Google Colab from disconnecting your runtime.
```

```markdown
### Step 1.2: Programmatic Dataset Download and Extraction
We will use a secure download strategy to retrieve the dataset files from Mendeley Data, extract them, and verify the physical directory structure.
```

In [2]:
import urllib.request
import zipfile
import os

# Mendeley direct download URL for this dataset version 1
dataset_url = "https://data.mendeley.com/public-files/datasets/j8z5cfcn33/files/b49d9c24-bca2-48df-9f4c-830efdb62124/file_downloaded"
output_zip = "mendeley_dataset.zip"
extract_dir = "mendeley_data_extracted"

try:
    print("Downloading dataset... Please wait as this might take a few moments.")
    urllib.request.urlretrieve(dataset_url, output_zip)
    print(f"Download complete. File saved as: {output_zip} ({os.path.getsize(output_zip) / (1024*1024):.2f} MB)")

    print("Extracting files...")
    with zipfile.ZipFile(output_zip, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    print(f"Extraction finished. Files extracted to: '{extract_dir}/'")

except Exception as e:
    print(f"Error encountered during download or extraction: {e}")
    print("Manual Action: If the Mendeley direct link changed, please manually upload the dataset zip file to Colab's file explorer on the left.")

Error encountered during download or extraction: HTTP Error 403: Forbidden
Manual Action: If the Mendeley direct link changed, please manually upload the dataset zip file to Colab's file explorer on the left.


```markdown
### Step 1.3: File System Audit and Formats Analysis
Now, let's crawl the extracted files to identify metadata, tabular data (CSV, XLSX, XLS), images, and other resources.

In [3]:
import glob

print("--- Extracted Files and Folders Structure ---")
all_files = []
for root, dirs, files in os.walk(extract_dir):
    for file in files:
        full_path = os.path.join(root, file)
        file_size_mb = os.path.getsize(full_path) / (1024 * 1024)
        all_files.append((full_path, file_size_mb))
        print(f"- {full_path} ({file_size_mb:.4f} MB)")

# Count file extensions
extensions = {}
for path, _ in all_files:
    ext = os.path.splitext(path)[1].lower()
    extensions[ext] = extensions.get(ext, 0) + 1

print("\n--- File Extension Breakdown ---")
for ext, count in extensions.items():
    print(f"{ext if ext else '[No Extension]'}: {count} file(s)")

--- Extracted Files and Folders Structure ---

--- File Extension Breakdown ---


```markdown
### Step 1.4: Tabular Data Inspection
We will now load any discovered tabular datasets (Excel/CSV) using Pandas to inspect column names, shapes, null counts, duplicate records, and target metrics.

In [4]:
import pandas as pd

# Locate any excel or csv files dynamically
tabular_files = [path for path, _ in all_files if os.path.splitext(path)[1].lower() in ['.csv', '.xlsx', '.xls']]

if not tabular_files:
    print("No standard tabular files (CSV or Excel) detected directly in the download.")
else:
    for file_path in tabular_files:
        print(f"\n{'='*50}\nInspecting: {file_path}\n{'='*50}")
        try:
            if file_path.endswith('.csv'):
                df = pd.read_csv(file_path)
            else:
                df = pd.read_excel(file_path)

            print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
            print("\nColumn Info and Missing Values:")
            missing_info = pd.DataFrame({
                'Dtype': df.dtypes,
                'Non-Null Count': df.notnull().sum(),
                'Missing Count': df.isnull().sum(),
                'Missing %': (df.isnull().sum() / len(df)) * 100
            })
            display(missing_info)

            duplicates = df.duplicated().sum()
            print(f"\nDuplicate rows detected: {duplicates}")

            print("\nFirst 5 Rows:")
            display(df.head())

            print("\nBasic Statistical Summary:")
            display(df.describe(include='all'))

        except Exception as e:
            print(f"Error reading {file_path}: {e}")

No standard tabular files (CSV or Excel) detected directly in the download.


```markdown
### Step 1.5: Image Folder and Metadata Verification
Let's evaluate whether there are images present (e.g. JPG, PNG), their dimensions, and check if any metadata text files accompany them to establish custom targets or ground-truth structural labels.

In [5]:
from PIL import Image

image_files = [path for path, _ in all_files if os.path.splitext(path)[1].lower() in ['.jpg', '.jpeg', '.png', '.bmp']]
print(f"Total images detected: {len(image_files)}")

if image_files:
    print("\nAnalyzing image sample properties:")
    for img_path in image_files[:5]:
        try:
            with Image.open(img_path) as img:
                print(f"- Image: {os.path.basename(img_path)} | Format: {img.format} | Size: {img.size} | Mode: {img.mode}")
        except Exception as e:
            print(f"Could not parse image {img_path}: {e}")
else:
    print("No raw image files detected.")

Total images detected: 0
No raw image files detected.


In [6]:

import requests
import json

dataset_id = "j8z5cfcn33"
version = 1

# Request metadata from the official Mendeley Data API
metadata_url = (
    f"https://api.data.mendeley.com/datasets/"
    f"{dataset_id}?version={version}"
)

print("Checking the official Mendeley dataset metadata...")

try:
    response = requests.get(
        metadata_url,
        headers={"Accept": "application/json"},
        timeout=30
    )

    print("HTTP status:", response.status_code)

    if response.status_code == 200:
        metadata = response.json()

        print("\nDataset metadata:")
        print(json.dumps(metadata, indent=2)[:15000])

    else:
        print("Metadata request was unsuccessful.")
        print(response.text[:2000])

except requests.RequestException as error:
    print("Request error:", error)

Checking the official Mendeley dataset metadata...
HTTP status: 401
Metadata request was unsuccessful.
{"message":"Unauthorized (no suitable authorization credentials were provided)"}


In [8]:

import os

print(os.listdir("/content"))

['.config', 'Predictive Structural Risk Evaluation Through Crack Imaging and Environmental Condition Metrics.zip', 'structural_crack_dataset_extended_500.xlsx', 'sample_data']


In [10]:
import os
import zipfile
import pandas as pd

# 1. Use absolute paths under /content/ for both uploaded files
excel_path = "/content/structural_crack_dataset_extended_500.xlsx"
zip_path = "/content/Predictive Structural Risk Evaluation Through Crack Imaging and Environmental Condition Metrics.zip"
extract_dir = "/content/extracted_crack_dataset"

print("=" * 60)
print("1. EXCEL DATASET INSPECTION")
print("=" * 60)
if os.path.exists(excel_path):
    df = pd.read_excel(excel_path)
    print(f"File: {excel_path}")
    print(f"Shape (Rows, Columns): {df.shape}")
    print(f"Exact Column Names: {list(df.columns)}")

    print("\n" + "=" * 60)
    print("2. MISSING VALUES & DUPLICATES")
    print("=" * 60)
    missing_counts = df.isnull().sum()
    if missing_counts.sum() > 0:
        print("Missing values per column:\n", missing_counts[missing_counts > 0])
    else:
        print("No missing values found across any column.")

    duplicates = df.duplicated().sum()
    print(f"Total duplicate rows: {duplicates}")

    print("\n" + "=" * 60)
    print("3. TARGET VARIABLE INSPECTION")
    print("=" * 60)
    print("Inspecting all columns and their unique value distributions to identify a valid target:")
    for col in df.columns:
        unique_vals = df[col].nunique()
        if unique_vals <= 20:  # Potential categorical or ordinal target
            print(f"Column '{col}' (dtype: {df[col].dtype}, unique count: {unique_vals}):")
            print(df[col].value_counts(dropna=False).to_dict())
        else:
            if pd.api.types.is_numeric_dtype(df[col]):
                print(f"Column '{col}' (dtype: {df[col].dtype}) -> Continuous (Range: {df[col].min()} to {df[col].max()})")
            else:
                print(f"Column '{col}' (dtype: {df[col].dtype}) -> High-cardinality text/identifier.")

    print("\n" + "=" * 60)
    print("4 & 5. ZIP EXTRACTION & IMAGE-TO-ROW MAPPING")
    print("=" * 60)

    # Check if already extracted
    if not os.path.exists(extract_dir) or not os.listdir(extract_dir):
        print("Extracting ZIP archive...")
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(extract_dir)
        print("Extraction complete.")
    else:
        print(f"ZIP archive already extracted at '{extract_dir}'. Skipping re-extraction.")

    # Gather all images preserving relative paths
    image_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff')
    extracted_images_rel = {} # relative path from extract_dir -> full path
    basename_counts = {}

    for root, dirs, files in os.walk(extract_dir):
        for file in files:
            if file.lower().endswith(image_extensions):
                rel_path = os.path.relpath(os.path.join(root, file), extract_dir)
                extracted_images_rel[rel_path] = os.path.join(root, file)
                base = os.path.basename(file)
                basename_counts[base] = basename_counts.get(base, 0) + 1

    print(f"Total valid image files found: {len(extracted_images_rel)}")

    duplicates_on_disk = {b: c for b, c in basename_counts.items() if c > 1}
    if duplicates_on_disk:
        print(f"Warning: Found duplicate basenames on disk: {duplicates_on_disk}")
    else:
        print("All image basenames on disk are unique.")

    # Dynamically find the image-reference column by checking which column values match image filenames or basenames
    best_match_col = None
    best_match_count = 0

    extracted_basenames = set(os.path.basename(p) for p in extracted_images_rel.keys())
    extracted_full_paths = set(extracted_images_rel.keys())

    for col in df.columns:
        col_vals = set(df[col].astype(str).str.strip())
        # Check both full relative path matches and filename/basename matches
        matches_full = len(col_vals.intersection(extracted_full_paths))
        matches_base = len(col_vals.intersection(extracted_basenames))
        max_matches = max(matches_full, matches_base)

        if max_matches > best_match_count:
            best_match_count = max_matches
            best_match_col = col

    print(f"\nIdentified Image-Reference Column: '{best_match_col}' (Matched {best_match_count}/{len(df)} rows)")

    if best_match_col:
        excel_refs = df[best_match_col].astype(str).str.strip()
        matched_rows = excel_refs.isin(extracted_full_paths) | excel_refs.isin(extracted_basenames)
        unmatched = df[~matched_rows]

        print(f"Exact Matches: {matched_rows.sum()}")
        print(f"Unmatched Rows: {len(unmatched)}")
        if len(unmatched) > 0:
            print("Unmatched examples:")
            print(unmatched[best_match_col].head(3))
    else:
        print("Could not automatically map any column to the extracted image filenames.")
else:
    print(f"Error: Excel file not found at {excel_path}")

1. EXCEL DATASET INSPECTION
File: /content/structural_crack_dataset_extended_500.xlsx
Shape (Rows, Columns): (291, 15)
Exact Column Names: ['ID', 'Image_Name', 'Structure_Type', 'Crack_Type', 'Crack_Length', 'Crack_Width', 'Crack_Depth', 'Material_Type', 'Age_of_Structure_yrs', 'Humidity_%', 'Moisture_%', 'Temperature_C', 'Crack_Location', 'Load_Bearing', 'Environment']

2. MISSING VALUES & DUPLICATES
No missing values found across any column.
Total duplicate rows: 0

3. TARGET VARIABLE INSPECTION
Inspecting all columns and their unique value distributions to identify a valid target:
Column 'ID' (dtype: int64) -> Continuous (Range: 1 to 291)
Column 'Image_Name' (dtype: object) -> High-cardinality text/identifier.
Column 'Structure_Type' (dtype: object, unique count: 8):
{'Wall': 48, 'column': 41, 'wall': 39, 'road': 36, 'beam': 35, 'House Pillar': 33, 'kerb wall': 31, 'Column': 28}
Column 'Crack_Type' (dtype: object, unique count: 4):
{'horizontal': 76, 'Horizontal': 73, 'vertical': 72

In [11]:

import os
import zipfile
import pandas as pd

# 1. Use absolute paths under /content/ for both uploaded files
excel_path = "/content/structural_crack_dataset_extended_500.xlsx"
zip_path = "/content/Predictive Structural Risk Evaluation Through Crack Imaging and Environmental Condition Metrics.zip"
extract_dir = "/content/extracted_crack_dataset"

print("=" * 60)
print("1. PRE-FLIGHT FILE & PATH VALIDATION")
print("=" * 60)

if not os.path.exists(excel_path):
    print(f"Error: Excel file not found at {excel_path}")
else:
    print(f"Excel file successfully located: {excel_path}")

if not os.path.exists(zip_path):
    print(f"Error: ZIP archive not found at {zip_path}")
else:
    print(f"ZIP archive successfully located: {zip_path}")

print("\n" + "=" * 60)
print("2. EXCEL DATASET INSPECTION")
print("=" * 60)
if os.path.exists(excel_path):
    df = pd.read_excel(excel_path)
    print(f"Shape (Rows, Columns): {df.shape}")
    print(f"Exact Column Names: {list(df.columns)}")

    print("\n" + "=" * 60)
    print("3. MISSING VALUES & DUPLICATES")
    print("=" * 60)
    missing_counts = df.isnull().sum()
    if missing_counts.sum() > 0:
        print("Missing values per column:\n", missing_counts[missing_counts > 0])
    else:
        print("No missing values found across any column.")

    duplicates = df.duplicated().sum()
    print(f"Total duplicate rows: {duplicates}")

    print("\n" + "=" * 60)
    print("4. COLUMN DISTRIBUTIONS & TARGET INSPECTION")
    print("=" * 60)
    print("Inspecting unique value distributions for all columns:")
    for col in df.columns:
        unique_vals = df[col].nunique()
        if unique_vals <= 20:
            print(f"Column '{col}' (dtype: {df[col].dtype}, unique count: {unique_vals}):")
            print(df[col].value_counts(dropna=False).to_dict())
        else:
            if pd.api.types.is_numeric_dtype(df[col]):
                print(f"Column '{col}' (dtype: {df[col].dtype}) -> Continuous (Range: {df[col].min()} to {df[col].max()})")
            else:
                print(f"Column '{col}' (dtype: {df[col].dtype}) -> High-cardinality text/identifier.")

    print("\n" + "=" * 60)
    print("5. ZIP EXTRACTION & IMAGE-TO-ROW MAPPING")
    print("=" * 60)

    if os.path.exists(zip_path):
        if not os.path.exists(extract_dir) or not os.listdir(extract_dir):
            print("Extracting ZIP archive...")
            with zipfile.ZipFile(zip_path, 'r') as z:
                z.extractall(extract_dir)
            print("Extraction complete.")
        else:
            print(f"ZIP archive already extracted at '{extract_dir}'. Skipping re-extraction.")

        # Gather all images preserving relative paths
        image_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff')
        extracted_images_rel = {}
        basename_counts = {}

        for root, dirs, files in os.walk(extract_dir):
            for file in files:
                if file.lower().endswith(image_extensions):
                    rel_path = os.path.relpath(os.path.join(root, file), extract_dir)
                    extracted_images_rel[rel_path] = os.path.join(root, file)
                    base = os.path.basename(file)
                    basename_counts[base] = basename_counts.get(base, 0) + 1

        print(f"Total valid image files found: {len(extracted_images_rel)}")

        duplicates_on_disk = {b: c for b, c in basename_counts.items() if c > 1}
        if duplicates_on_disk:
            print(f"Warning: Found duplicate basenames on disk: {duplicates_on_disk}")
        else:
            print("All image basenames on disk are unique.")

        # Dynamically find the image-reference column
        best_match_col = None
        best_match_count = 0

        extracted_basenames = set(os.path.basename(p) for p in extracted_images_rel.keys())
        extracted_full_paths = set(extracted_images_rel.keys())

        for col in df.columns:
            col_vals = set(df[col].astype(str).str.strip())
            matches_full = len(col_vals.intersection(extracted_full_paths))
            matches_base = len(col_vals.intersection(extracted_basenames))
            max_matches = max(matches_full, matches_base)

            if max_matches > best_match_count:
                best_match_count = max_matches
                best_match_col = col

        print(f"\nIdentified Image-Reference Column: '{best_match_col}' (Matched {best_match_count}/{len(df)} rows)")

        if best_match_col:
            excel_refs = df[best_match_col].astype(str).str.strip()
            matched_rows = excel_refs.isin(extracted_full_paths) | excel_refs.isin(extracted_basenames)
            unmatched = df[~matched_rows]

            print(f"Exact Matches: {matched_rows.sum()}")
            print(f"Unmatched Rows: {len(unmatched)}")
            if len(unmatched) > 0:
                print("Unmatched examples:")
                print(unmatched[best_match_col].head(3))
        else:
            print("Could not automatically map any column to the extracted image filenames.")
    else:
        print(f"Skipping image mapping because ZIP file was not found at {zip_path}")
else:
    print("Execution aborted because the Excel file could not be loaded.")

1. PRE-FLIGHT FILE & PATH VALIDATION
Excel file successfully located: /content/structural_crack_dataset_extended_500.xlsx
ZIP archive successfully located: /content/Predictive Structural Risk Evaluation Through Crack Imaging and Environmental Condition Metrics.zip

2. EXCEL DATASET INSPECTION
Shape (Rows, Columns): (291, 15)
Exact Column Names: ['ID', 'Image_Name', 'Structure_Type', 'Crack_Type', 'Crack_Length', 'Crack_Width', 'Crack_Depth', 'Material_Type', 'Age_of_Structure_yrs', 'Humidity_%', 'Moisture_%', 'Temperature_C', 'Crack_Location', 'Load_Bearing', 'Environment']

3. MISSING VALUES & DUPLICATES
No missing values found across any column.
Total duplicate rows: 0

4. COLUMN DISTRIBUTIONS & TARGET INSPECTION
Inspecting unique value distributions for all columns:
Column 'ID' (dtype: int64) -> Continuous (Range: 1 to 291)
Column 'Image_Name' (dtype: object) -> High-cardinality text/identifier.
Column 'Structure_Type' (dtype: object, unique count: 8):
{'Wall': 48, 'column': 41, 'wa

In [12]:

import os
import pandas as pd

excel_path = "/content/structural_crack_dataset_extended_500.xlsx"
image_dir = "/content/extracted_crack_dataset"

df = pd.read_excel(excel_path)

print("FIRST 15 IMAGE NAMES IN EXCEL:")
print(df["Image_Name"].head(15).to_list())

print("\nFIRST 20 IMAGE FILES IN THE ZIP:")
image_files = []
for root, dirs, files in os.walk(image_dir):
    for file in files:
        if file.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".tiff")):
            image_files.append(os.path.relpath(
                os.path.join(root, file), image_dir
            ))

print(sorted(image_files)[:20])
print("\nTotal images:", len(image_files))

FIRST 15 IMAGE NAMES IN EXCEL:
['img_001.jpg', 'img_002.jpg', 'img_003.jpg', 'img_004.jpg', 'img_005.jpg', 'img_006.jpg', 'img_007.jpg', 'img_008.jpg', 'img_009.jpg', 'img_010.jpg', 'img_011.jpg', 'img_012.jpg', 'img_013.jpg', 'img_014.jpg', 'img_015.jpg']

FIRST 20 IMAGE FILES IN THE ZIP:
['Images/001.jpg', 'Images/002.jpg', 'Images/003.jpg', 'Images/004.jpg', 'Images/005.jpg', 'Images/006.jpg', 'Images/007.jpg', 'Images/008.jpg', 'Images/009.jpg', 'Images/010.jpg', 'Images/011.jpg', 'Images/012.jpg', 'Images/013.jpg', 'Images/014.jpg', 'Images/015.jpg', 'Images/016.jpg', 'Images/017.jpg', 'Images/018.jpg', 'Images/019.jpg', 'Images/100.jpg']

Total images: 290


In [13]:

import os
import pandas as pd

excel_path = "/content/structural_crack_dataset_extended_500.xlsx"
image_dir = "/content/extracted_crack_dataset"

df = pd.read_excel(excel_path)

# Create a lookup using the actual image filenames
image_lookup = {}

for root, dirs, files in os.walk(image_dir):
    for file in files:
        if file.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".tiff")):
            image_lookup[file.lower()] = os.path.join(root, file)

# Convert img_001.jpg into 001.jpg for matching
df["Expected_Image_Name"] = (
    df["Image_Name"].astype(str).str.strip()
    .str.replace(r"^img_", "", regex=True)
    .str.lower()
)

df["Image_Matched"] = df["Expected_Image_Name"].isin(image_lookup)

print("Total Excel rows:", len(df))
print("Successfully matched images:", int(df["Image_Matched"].sum()))
print("Rows without matching images:", int((~df["Image_Matched"]).sum()))

print("\nFirst unmatched rows:")
print(df.loc[~df["Image_Matched"], ["ID", "Image_Name"]].head(10))

print("\nFirst 5 successful matches:")
for _, row in df[df["Image_Matched"]].head(5).iterrows():
    print(row["Image_Name"], "->", image_lookup[row["Expected_Image_Name"]])

Total Excel rows: 291
Successfully matched images: 210
Rows without matching images: 81

First unmatched rows:
    ID   Image_Name
19  20  img_020.jpg
20  21  img_021.jpg
21  22  img_022.jpg
22  23  img_023.jpg
23  24  img_024.jpg
24  25  img_025.jpg
25  26  img_026.jpg
26  27  img_027.jpg
27  28  img_028.jpg
28  29  img_029.jpg

First 5 successful matches:
img_001.jpg -> /content/extracted_crack_dataset/Images/001.jpg
img_002.jpg -> /content/extracted_crack_dataset/Images/002.jpg
img_003.jpg -> /content/extracted_crack_dataset/Images/003.jpg
img_004.jpg -> /content/extracted_crack_dataset/Images/004.jpg
img_005.jpg -> /content/extracted_crack_dataset/Images/005.jpg


In [14]:

import os
import re

image_dir = "/content/extracted_crack_dataset"

# Collect image numbers from the actual files
found_numbers = set()

for root, dirs, files in os.walk(image_dir):
    for file in files:
        match = re.fullmatch(r"(\d+)\.(jpg|jpeg|png|bmp|tiff)", file, re.I)
        if match:
            found_numbers.add(int(match.group(1)))

expected_numbers = set(range(1, 292))
missing_numbers = sorted(expected_numbers - found_numbers)
extra_numbers = sorted(found_numbers - expected_numbers)

print("Image numbers found:", len(found_numbers))
print("Missing image numbers:", len(missing_numbers))
print("First 100 missing numbers:", missing_numbers[:100])
print("Image numbers outside 1-291:", extra_numbers[:30])

print("\nFiles around the first gap:")
for number in range(15, 36):
    filename = f"{number:03d}.jpg"
    matches = []
    for root, dirs, files in os.walk(image_dir):
        if filename in files:
            matches.append(os.path.join(root, filename))
    print(filename, "FOUND" if matches else "MISSING")

Image numbers found: 289
Missing image numbers: 2
First 100 missing numbers: [39, 277]
Image numbers outside 1-291: []

Files around the first gap:
015.jpg FOUND
016.jpg FOUND
017.jpg FOUND
018.jpg FOUND
019.jpg FOUND
020.jpg MISSING
021.jpg MISSING
022.jpg MISSING
023.jpg MISSING
024.jpg MISSING
025.jpg MISSING
026.jpg MISSING
027.jpg MISSING
028.jpg MISSING
029.jpg MISSING
030.jpg MISSING
031.jpg MISSING
032.jpg MISSING
033.jpg MISSING
034.jpg MISSING
035.jpg MISSING


In [15]:

import zipfile
from collections import Counter

zip_path = "/content/Predictive Structural Risk Evaluation Through Crack Imaging and Environmental Condition Metrics.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    files = [name for name in z.namelist() if not name.endswith("/")]

print("Total files inside ZIP:", len(files))

print("\nFirst 50 file paths:")
for name in files[:50]:
    print(name)

print("\nFile extensions:")
extensions = Counter(
    name.rsplit(".", 1)[-1].lower()
    for name in files if "." in name
)
print(extensions)

print("\nPaths containing 020 or 039 or 277:")
for name in files:
    if any(x in name.lower() for x in ["020", "039", "277"]):
        print(name)

Total files inside ZIP: 290

First 50 file paths:
Images/274.jpg
Images/287.jpg
Images/269.jpg
Images/286.jpg
Images/259.jpg
Images/265.jpg
Images/278.jpg
Images/272.jpg
Images/283.jpg
Images/268.jpg
Images/279.jpg
Images/291.jpg
Images/273.jpg
Images/264.jpg
Images/284.jpg
Images/267.jpg
Images/271.jpg
Images/261.jpg
Images/289.jpg
Images/250.jpg
Images/282.jpg
Images/253.jpg
Images/270.jpg
Images/254.jpg
Images/247.jpg
Images/260.jpg
Images/251.jpg
Images/281.jpg
Images/266.jpg
Images/285.jpg
Images/241.jpg
Images/263.jpg
Images/257.jpg
Images/242.jpg
Images/288.jpg
Images/248.jpg
Images/239.jpg
Images/231.jpg
Images/252.jpg
Images/232.jpg
Images/280.jpg
Images/243.jpg
Images/275.jpg
Images/227.jpg
Images/225.jpg
Images/224.jpg
Images/258.jpg
Images/237.jpg
Images/240.jpg
Images/246.jpg

File extensions:
Counter({'jpg': 290})

Paths containing 020 or 039 or 277:


In [16]:

import zipfile
import os
import re

zip_path = "/content/Predictive Structural Risk Evaluation Through Crack Imaging and Environmental Condition Metrics.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    paths = [
        name for name in z.namelist()
        if name.lower().endswith(".jpg")
    ]

numbers = []
invalid_names = []

for path in paths:
    filename = os.path.basename(path)
    match = re.fullmatch(r"(\d+)\.jpg", filename, re.I)

    if match:
        numbers.append(int(match.group(1)))
    else:
        invalid_names.append(filename)

unique_numbers = set(numbers)
missing_numbers = sorted(set(range(1, 292)) - unique_numbers)
duplicate_numbers = sorted(
    number for number in unique_numbers
    if numbers.count(number) > 1
)

print("Total JPG files:", len(paths))
print("Files with numeric filenames:", len(numbers))
print("Unique image numbers:", len(unique_numbers))
print("Missing numbers from 1 to 291:", missing_numbers)
print("Duplicate image numbers:", duplicate_numbers)
print("Non-numeric JPG filenames:", invalid_names[:20])
print("Example filenames:", [os.path.basename(p) for p in paths[:10]])

Total JPG files: 290
Files with numeric filenames: 289
Unique image numbers: 289
Missing numbers from 1 to 291: [39, 277]
Duplicate image numbers: []
Non-numeric JPG filenames: ['181(1).jpg']
Example filenames: ['274.jpg', '287.jpg', '269.jpg', '286.jpg', '259.jpg', '265.jpg', '278.jpg', '272.jpg', '283.jpg', '268.jpg']


In [17]:

import os
import zipfile
import pandas as pd

excel_path = "/content/structural_crack_dataset_extended_500.xlsx"
zip_path = "/content/Predictive Structural Risk Evaluation Through Crack Imaging and Environmental Condition Metrics.zip"
extract_dir = "/content/extracted_crack_dataset"

df = pd.read_excel(excel_path)

with zipfile.ZipFile(zip_path, "r") as z:
    zip_files = [
        name for name in z.namelist()
        if name.lower().endswith(".jpg")
    ]

# Build lookup by exact numeric filename
image_lookup = {}
for path in zip_files:
    filename = os.path.basename(path)
    if filename.lower().endswith(".jpg"):
        stem = filename[:-4]
        if stem.isdigit():
            image_lookup[int(stem)] = path

# Match Excel names such as img_001.jpg to 001.jpg
df["Image_Number"] = (
    df["Image_Name"].astype(str)
    .str.extract(r"(\d+)", expand=False)
    .astype("Int64")
)

df["Image_Path_In_ZIP"] = df["Image_Number"].map(image_lookup)
df["Image_Matched"] = df["Image_Path_In_ZIP"].notna()

print("Excel rows:", len(df))
print("Rows with matched images:", int(df["Image_Matched"].sum()))
print("Rows without matched images:", int((~df["Image_Matched"]).sum()))

print("\nUnmatched Excel rows:")
print(df.loc[~df["Image_Matched"], ["ID", "Image_Name"]].to_string(index=False))

print("\nSpecial filename:")
print("181(1).jpg exists:", any(os.path.basename(p) == "181(1).jpg" for p in zip_files))

Excel rows: 291
Rows with matched images: 289
Rows without matched images: 2

Unmatched Excel rows:
 ID  Image_Name
 39 img_039.jpg
277 img_277.jpg

Special filename:
181(1).jpg exists: True


In [18]:

import pandas as pd

file_path = "/content/structural_crack_dataset_extended_500.xlsx"
df = pd.read_excel(file_path)

print("CRACK WIDTH DISTRIBUTION")
print(df["Crack_Width"].value_counts().sort_index())

print("\nCRACK DEPTH DISTRIBUTION")
print(df["Crack_Depth"].value_counts().sort_index())

print("\nROWS WITH WIDTH OR DEPTH ABOVE 10")
unusual = df[
    (df["Crack_Width"] > 10) |
    (df["Crack_Depth"] > 10)
]

print(
    unusual[
        ["ID", "Crack_Width", "Crack_Depth", "Crack_Type"]
    ].to_string(index=False)
)

print("\nAVAILABLE COLUMNS")
print(df.columns.tolist())

CRACK WIDTH DISTRIBUTION
Crack_Width
1      51
2      57
3      69
4      65
5      47
9       1
100     1
Name: count, dtype: int64

CRACK DEPTH DISTRIBUTION
Crack_Depth
1      36
2      29
3      36
4      33
5      32
6      37
7      29
8      30
9      28
150     1
Name: count, dtype: int64

ROWS WITH WIDTH OR DEPTH ABOVE 10
 ID  Crack_Width  Crack_Depth Crack_Type
 15          100          150 horizontal

AVAILABLE COLUMNS
['ID', 'Image_Name', 'Structure_Type', 'Crack_Type', 'Crack_Length', 'Crack_Width', 'Crack_Depth', 'Material_Type', 'Age_of_Structure_yrs', 'Humidity_%', 'Moisture_%', 'Temperature_C', 'Crack_Location', 'Load_Bearing', 'Environment']


In [19]:

import pandas as pd

path = "/content/structural_crack_dataset_extended_500.xlsx"
df = pd.read_excel(path)

columns = [
    "ID", "Image_Name", "Structure_Type", "Crack_Type",
    "Crack_Length", "Crack_Width", "Crack_Depth",
    "Material_Type", "Age_of_Structure_yrs",
    "Humidity_%", "Moisture_%", "Temperature_C",
    "Crack_Location", "Load_Bearing", "Environment"
]

print("SUSPICIOUS RECORD — ID 15")
print(df.loc[df["ID"] == 15, columns].to_string(index=False))

print("\nCOMPARISON — FIVE ORDINARY RECORDS")
print(
    df.loc[df["ID"].isin([14, 16, 17, 18, 19]), columns]
      .to_string(index=False)
)

SUSPICIOUS RECORD — ID 15
 ID  Image_Name Structure_Type Crack_Type  Crack_Length  Crack_Width  Crack_Depth Material_Type  Age_of_Structure_yrs  Humidity_%  Moisture_%  Temperature_C Crack_Location Load_Bearing Environment
 15 img_015.jpg           road horizontal          4500          100          150       bitumen                     2   49.091248    6.301032             40         corner           no       urban

COMPARISON — FIVE ORDINARY RECORDS
 ID  Image_Name Structure_Type Crack_Type  Crack_Length  Crack_Width  Crack_Depth Material_Type  Age_of_Structure_yrs  Humidity_%  Moisture_%  Temperature_C Crack_Location Load_Bearing Environment
 14 img_014.jpg           beam horizontal           356            4            6      concrete                     9   50.616956    8.410482             31       top Edge          yes       urban
 16 img_016.jpg           beam horizontal           452            3            2      concrete                     9   49.170225   23.977711         

In [21]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the dataset
df = pd.read_excel(
    "/content/structural_crack_dataset_extended_500.xlsx"
)

# Split the data
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42
)

# Create an exploratory crack-severity proxy
dimensions = ["Crack_Length", "Crack_Width", "Crack_Depth"]

scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_df[dimensions])
test_scaled = scaler.transform(test_df[dimensions])

train_score = train_scaled.mean(axis=1)
test_score = test_scaled.mean(axis=1)

threshold = np.percentile(train_score, 75)

y_train = (train_score >= threshold).astype(int)
y_test = (test_score >= threshold).astype(int)

# Use other columns to predict the proxy
features = [
    "Structure_Type", "Crack_Type", "Material_Type",
    "Age_of_Structure_yrs", "Humidity_%", "Moisture_%",
    "Temperature_C", "Crack_Location", "Load_Bearing",
    "Environment"
]

X_train = train_df[features].copy()
X_test = test_df[features].copy()

print("Preparation successful!")
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training labels:", np.unique(y_train, return_counts=True))



Preparation successful!
Training rows: 232
Testing rows: 59
Training labels: (array([0, 1]), array([174,  58]))


In [22]:

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Separate categorical and numerical features
categorical_features = [
    "Structure_Type",
    "Crack_Type",
    "Material_Type",
    "Crack_Location",
    "Load_Bearing",
    "Environment"
]

numerical_features = [
    "Age_of_Structure_yrs",
    "Humidity_%",
    "Moisture_%",
    "Temperature_C"
]

# Prepare the data
preprocessor = ColumnTransformer([
    ("categorical", OneHotEncoder(handle_unknown="ignore"),
     categorical_features),
    ("numerical", StandardScaler(), numerical_features)
])

# Build the model
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=42
    ))
])

# Train
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Evaluate
print("MODEL TRAINING COMPLETED!")
print("\nAccuracy:", round(accuracy_score(y_test, y_pred), 3))
print("Precision:", round(precision_score(y_test, y_pred, zero_division=0), 3))
print("Recall:", round(recall_score(y_test, y_pred, zero_division=0), 3))
print("F1 Score:", round(f1_score(y_test, y_pred, zero_division=0), 3))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))


MODEL TRAINING COMPLETED!

Accuracy: 0.729
Precision: 0.5
Recall: 0.062
F1 Score: 0.111

Confusion Matrix:
[[42  1]
 [15  1]]

Classification Report:
              precision    recall  f1-score   support

           0       0.74      0.98      0.84        43
           1       0.50      0.06      0.11        16

    accuracy                           0.73        59
   macro avg       0.62      0.52      0.48        59
weighted avg       0.67      0.73      0.64        59



In [23]:

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix
)

# Reuse the preprocessing pipeline from your existing model
improved_model = clone(model)

# Give both classes equal importance
improved_model.set_params(
    classifier__class_weight="balanced_subsample",
    classifier__min_samples_leaf=2
)

# Train the improved model
improved_model.fit(X_train, y_train)

# Get probabilities for the higher-severity proxy class
probabilities = improved_model.predict_proba(X_test)[:, 1]

# Lower threshold to try detecting more positive cases
new_threshold = 0.30
improved_predictions = (probabilities >= new_threshold).astype(int)

print("IMPROVED MODEL RESULTS")
print("Threshold:", new_threshold)
print("Accuracy:", round(accuracy_score(y_test, improved_predictions), 3))
print("Precision:", round(precision_score(
    y_test, improved_predictions, zero_division=0
), 3))
print("Recall:", round(recall_score(
    y_test, improved_predictions, zero_division=0
), 3))
print("F1 Score:", round(f1_score(
    y_test, improved_predictions, zero_division=0
), 3))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, improved_predictions))

print("\nClassification Report:")
print(classification_report(y_test, improved_predictions, zero_division=0))

IMPROVED MODEL RESULTS
Threshold: 0.3
Accuracy: 0.542
Precision: 0.333
Recall: 0.688
F1 Score: 0.449

Confusion Matrix:
[[21 22]
 [ 5 11]]

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.49      0.61        43
           1       0.33      0.69      0.45        16

    accuracy                           0.54        59
   macro avg       0.57      0.59      0.53        59
weighted avg       0.68      0.54      0.57        59



In [24]:

import joblib
import pandas as pd
from sklearn.metrics import classification_report

# Save the improved model
joblib.dump(
    improved_model,
    "/content/crack_severity_model.pkl"
)

# Save predictions and actual proxy labels
results = pd.DataFrame({
    "Actual_Severity_Proxy": y_test,
    "Predicted_Severity_Proxy": improved_predictions,
    "Predicted_Probability": probabilities
})

results.to_csv(
    "/content/model_predictions.csv",
    index=False
)

# Save evaluation report
report = classification_report(
    y_test,
    improved_predictions,
    output_dict=True,
    zero_division=0
)

pd.DataFrame(report).transpose().to_csv(
    "/content/model_evaluation.csv"
)

print("Saved successfully!")
print("1. crack_severity_model.pkl")
print("2. model_predictions.csv")
print("3. model_evaluation.csv")


Saved successfully!
1. crack_severity_model.pkl
2. model_predictions.csv
3. model_evaluation.csv


In [26]:

from google.colab import files

files.download("/content/crack_severity_model.pkl")
files.download("/content/model_predictions.csv")
files.download("/content/model_evaluation.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>